# 05 — From token IDs to embeddings (Part F)

NumPy exercises first, then you package them in `src/embeddings.py` (`pytest tests/test_embeddings.py`).

**How the 🧩 exercises work.** Cells marked 🧩 contain `____` blanks: complete them using the hints (they name the Python functions you need). Each exercise is followed by a ✅ *check* cell that tells you immediately whether you got it right. When an exercise matches a function of `src/`, the docstring of that function says so: you then **move your working code into `src/`** and run `pytest`.

In [ ]:
import sys, pathlib, json, time
ROOT = pathlib.Path.cwd().parent          # notebooks/ is one level below the repo root
sys.path.insert(0, str(ROOT))
import pandas as pd
import matplotlib.pyplot as plt
%load_ext autoreload
%autoreload 2

import numpy as np
from src.embeddings import EmbeddingMatrix
from src.bpe import BPETokenizer

## F.1 Shapes and parameter counts (theory)

Let V = 10 000 and d = 256. Answer **before** running anything:

1. Shape of E? 2. Number of trainable parameters? 3. Shape of the id sequence for a 12-token sentence? 4. Shape of its embedding? 5. How does the layer retrieve the vector of a token id? 6. Why is E trainable?

**Your answer:**

_(write here)_

### 🧩 Exercise 5.1 — An embedding matrix and a lookup

Create a small embedding matrix `E` (V = 10 tokens, d = 4) with random values, and look up the vectors of the ids `[3, 1, 3]`.

Replace every `____` with the right code, run the cell, then run the **check** cell below it.

**Hints**
* `rng = np.random.default_rng(0)`; `rng.normal(mean, std, size=(rows, cols))`.
* NumPy *fancy indexing*: `E[ids]` with an integer array selects rows and returns shape `ids.shape + (d,)`.
* `E.shape` is `(V, d)`, `E.size` the number of entries (= number of parameters).

In [ ]:
rng = np.random.default_rng(0)
V, d = 10, 4
E = rng.____(0, 0.02, size=____)

ids = np.array([3, 1, 3])
vectors = E[____]
print(E.shape, E.size, vectors.shape)

In [ ]:
assert E.shape == (10, 4) and E.size == 40 and vectors.shape == (3, 4)
assert np.array_equal(vectors[0], E[3]) and np.array_equal(vectors[0], vectors[2])
print("✅ Exercise 5.1 OK")

### 🧩 Exercise 5.2 — The lookup is a matrix product

Build the one-hot matrix of the ids and show that `one_hot @ E` gives exactly the same vectors.

Replace every `____` with the right code, run the cell, then run the **check** cell below it.

**Hints**
* `np.eye(V)` is the V×V identity matrix: its row `i` is the one-hot vector of id `i`. So `np.eye(V)[ids]` is the one-hot matrix of a whole sequence.
* Matrix product in NumPy: `A @ B`.
* Compare floating-point arrays with `np.allclose`.

In [ ]:
one_hot = np.____(V)[____]
via_matmul = one_hot ____ E
print(one_hot.shape, via_matmul.shape, np.allclose(via_matmul, vectors))

In [ ]:
assert one_hot.shape == (3, 10) and one_hot.sum() == 3
assert np.allclose(via_matmul, vectors)
print("✅ Exercise 5.2 OK")

### 🧩 Exercise 5.3 — Batches

Look up a *batch* of 2 sentences of 5 tokens each, and count the parameters of a matrix with V = 50 and d = 8.

Replace every `____` with the right code, run the cell, then run the **check** cell below it.

**Hints**
* A 2-D integer array `ids` of shape `(batch, length)`; `E[ids]` then has shape `(batch, length, d)`.
* Parameters of a V×d matrix: `V * d`.

In [ ]:
batch_ids = np.array([[1, 2, 3, 4, 5], [5, 4, 3, 2, 1]])
batch_vectors = E[____]
params = ____
print(batch_ids.shape, "->", batch_vectors.shape, "| parameters (V=50, d=8):", params)

In [ ]:
assert batch_vectors.shape == (2, 5, 4) and params == 400
print("✅ Exercise 5.3 OK")

## ⏸ Stop: package your code in `src/embeddings.py`

| Exercise | → `EmbeddingMatrix` |
|---|---|
| 5.1 | `__call__` (+ the out-of-range check: `IndexError` for ids < 0 or ≥ V) |
| 5.2 | `one_hot` and `lookup_via_matmul` |

In [ ]:
E = EmbeddingMatrix(vocab_size=10_000, embedding_dim=256)
ids = np.arange(12)
print("E:", E.weight.shape, "| parameters:", f"{E.num_parameters:,}")
print("ids:", ids.shape, "| embeddings:", E(ids).shape)
print("batch of 1 sentence of 5 tokens:", E([[12, 45, 78, 91, 103]]).shape)

**Explain** the output shapes, and why real frameworks use a lookup instead of the matrix product (compare the cost for V = 10 000, d = 256, T = 12).

**Your answer:**

_(write here)_

## F.3 Connect it to your BPE tokenizer

In [ ]:
toy = (ROOT / "data/toy/toy_corpus.txt").read_text(encoding="utf-8").splitlines()
bpe = BPETokenizer().train(toy, num_merges=10)
emb = EmbeddingMatrix(bpe.vocab_size, embedding_dim=4)
tokens = bpe.encode("low lower"); ids = bpe.tokens_to_ids(tokens)
print(tokens, ids); print(emb(ids).shape)

**Questions**
1. How many parameters would the embedding matrix have for your BPE-1000 vocabulary (notebook 04) with d = 256? For the character vocabulary of notebook 02?
2. Using notebooks 02 and 04, what is the trade-off between a large vocabulary (big E) and long sequences?

**Your answer:**

_(write here)_